# Module 01 · IC và mạch logic (EASA M04 SM01)

## 1. Đổi nhị phân ↔ thập phân

In [1]:
for d in (13, 29, 45, 100, 255): print(d, "->", format(d, "b"))
for b in ("1011", "11001", "100101"): print(b, "->", int(b, 2))
print("5 + 3 =", format(5+3, "b"), "; 13 + 3 =", format(13+3, "b"))

13 -> 1101
29 -> 11101
45 -> 101101
100 -> 1100100
255 -> 11111111
1011 -> 11
11001 -> 25
100101 -> 37
5 + 3 = 1000 ; 13 + 3 = 10000


**Nhận xét:** 13 → 1101, 45 → 101101, 255 → 11111111; 1011₂ = 11, 11001₂ = 25; 5 + 3 = 1000₂ và 13 + 3 = 10000₂ (nhớ lan qua nhiều cột).

## 2. Bảng chân lý các cổng

In [2]:
import itertools
G = {"AND": lambda a,b: a&b, "OR": lambda a,b: a|b, "NAND": lambda a,b: 1-(a&b), "NOR": lambda a,b: 1-(a|b), "XOR": lambda a,b: a^b}
print("A B | " + " ".join(f"{n:>4}" for n in G))
for a, b in itertools.product((0,1), repeat=2): print(a, b, "| " + " ".join(f"{G[n](a,b):>4}" for n in G))
print("De Morgan check (NOR == AND of inverted inputs):", all(G["NOR"](a,b) == ((1-a)&(1-b)) for a,b in itertools.product((0,1), repeat=2)))
print("De Morgan check (NAND == OR of inverted inputs):", all(G["NAND"](a,b) == ((1-a)|(1-b)) for a,b in itertools.product((0,1), repeat=2)))

A B |  AND   OR NAND  NOR  XOR
0 0 |    0    0    1    1    0
0 1 |    0    1    1    0    1
1 0 |    0    1    1    0    1
1 1 |    1    1    0    0    0
De Morgan check (NOR == AND of inverted inputs): True
De Morgan check (NAND == OR of inverted inputs): True


**Nhận xét:** XOR chỉ khác OR ở dòng 1-1 (0 thay vì 1); NOR bằng AND của các ngõ vào đảo và NAND bằng OR của các ngõ vào đảo (cả hai kiểm tra đều True) — đó là lý do Negative AND ≡ NOR, Negative OR ≡ NAND.

## 3. Bộ cộng nửa và bộ cộng đầy đủ

In [3]:
def half(a, b): return a ^ b, a & b
def full(a, b, cin):
    s1, c1 = half(a, b); s, c2 = half(s1, cin); return s, c1 | c2
for a in (0,1):
    for b in (0,1): print(f"half({a},{b}) -> (sum,carry) = {half(a,b)}")
A, B = 0b1101, 0b0011; carry = 0; out = []
for i in range(4):
    s, carry = full((A>>i)&1, (B>>i)&1, carry); out.append(s)
print("ripple add 1101 + 0011 =", carry, "".join(map(str, reversed(out))))

half(0,0) -> (sum,carry) = (0, 0)
half(0,1) -> (sum,carry) = (1, 0)
half(1,0) -> (sum,carry) = (1, 0)
half(1,1) -> (sum,carry) = (0, 1)
ripple add 1101 + 0011 = 1 0000


**Nhận xét:** bộ cộng nửa cho (0,0), (1,0), (1,0), (0,1); cộng 1101 + 0011 bằng chuỗi bộ cộng đầy đủ cho nhớ ra 1 và tổng 0000, tức 10000₂ = 16.

## 4. JK flip-flop

In [4]:
def jk(q, j, k): return {(0,0): q, (0,1): 0, (1,0): 1, (1,1): 1-q}[(j,k)]
for j, k in ((0,0),(0,1),(1,0),(1,1)): print(f"J={j} K={k}: Q=0 -> {jk(0,j,k)},  Q=1 -> {jk(1,j,k)}")
q = 0; seq = []
for _ in range(6): q = jk(q, 1, 1); seq.append(q)
print("J=K=1 over 6 clocks:", seq, "-> divide-by-2 of the clock")

J=0 K=0: Q=0 -> 0,  Q=1 -> 1
J=0 K=1: Q=0 -> 0,  Q=1 -> 0
J=1 K=0: Q=0 -> 1,  Q=1 -> 1
J=1 K=1: Q=0 -> 1,  Q=1 -> 0
J=K=1 over 6 clocks: [1, 0, 1, 0, 1, 0] -> divide-by-2 of the clock


**Nhận xét:** J = K = 1 làm Q đảo mỗi xung nhịp (1,0,1,0,1,0) nên flip-flop chia tần số xung nhịp cho 2 — nền tảng của bộ đếm.